In [ ]:
import pandas as pd
import jieba
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score

In [ ]:
train_data = pd.read_csv('data/train.tsv', sep='\t')
valid_data = pd.read_csv('data/dev.tsv', sep='\t')
test_data = pd.read_csv('data/test.tsv', sep='\t') 
x_train, y_train = train_data.text_a.values, train_data.label.values # ???
x_valid, y_valid = valid_data.text_a.values, valid_data.label.values # ???
x_test, y_test = test_data.text_a.values, test_data.label.values # ???

In [ ]:
# ??
from sklearn.metrics import classification_report

def cut_words(text):
    return ' '.join(jieba.cut(text))

x_train = [cut_words(text) for text in x_train]
x_valid = [cut_words(text) for text in x_valid]
x_test = [cut_words(text) for text in x_test]

# tf-idf
vectorizer = TfidfVectorizer()
x_train = vectorizer.fit_transform(x_train)
x_valid = vectorizer.transform(x_valid)
x_test = vectorizer.transform(x_test)


In [ ]:
# use other ml model
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
# use more model
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF

In [ ]:
# SVM
print("svm:")
clf_SVC = SVC(kernel='linear')
clf_SVC.fit(x_train, y_train)
report_svm = classification_report(y_test, clf_SVC.predict(x_test),digits=4)
print(report_svm)
print("___________________________________________________")

In [ ]:
# Naive Bayes
print("naive bayes:")
clf_NB = MultinomialNB()
clf_NB.fit(x_train, y_train)
report_nb = classification_report(y_test, clf_NB.predict(x_test),digits=4)
print(report_nb)
print("___________________________________________________")

In [ ]:
# Logistic Regression
print("losgistic regression:")
clf_LR = LogisticRegression()
clf_LR.fit(x_train, y_train)
report_lr = classification_report(y_test, clf_LR.predict(x_test),digits=4)
print(report_lr)
print("___________________________________________________")

In [ ]:
# Random Forest
print("random forest:")
clf_RF = RandomForestClassifier()
clf_RF.fit(x_train, y_train)
report_rf = classification_report(y_test, clf_RF.predict(x_test),digits=4)
print(report_rf)
print("___________________________________________________")

In [ ]:
# Decision Tree
print("decision tree:")
clf_DT = DecisionTreeClassifier()
clf_DT.fit(x_train, y_train)
report_dt = classification_report(y_test, clf_DT.predict(x_test),digits=4)
print(report_dt)
print("___________________________________________________")

In [ ]:
# AdaBoost
print("adaboost:")
clf_AdB = AdaBoostClassifier()
clf_AdB.fit(x_train, y_train)
report_ada = classification_report(y_test, clf_AdB.predict(x_test),digits=4)
print(report_ada)
print("___________________________________________________")


In [ ]:

# GradientBoosting
print("gradient boosting:")
clf_GB = GradientBoostingClassifier()
clf_GB.fit(x_train, y_train)
report_gb = classification_report(y_test, clf_GB.predict(x_test),digits=4)
print(report_gb)
print("___________________________________________________")


In [ ]:
# SGD
print("sgd:")
clf_SGD = SGDClassifier()
clf_SGD.fit(x_train, y_train)
report_sgd = classification_report(y_test, clf_SGD.predict(x_test),digits=4)
print(report_sgd)
print("___________________________________________________")


In [ ]:
# Bagging
print("bagging:")
clf_BG = BaggingClassifier()
clf_BG.fit(x_train, y_train)
report_bag = classification_report(y_test, clf_BG.predict(x_test),digits=4)
print(report_bag)
print("___________________________________________________")

In [ ]:
# get the information of every report from above like precision, recall, f1-score, accuracy and make a table
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def get_report(report):
    report = report.split("\n")
    res = []
    for i in report:
        i = re.sub(" +", " ", i.strip())
        temp = i.split(" ")
        res.append(temp[1:])

    return res

def get_df(report):
    res = get_report(report)
    acc = float(res[5][0])
    precision = float(res[6][1])
    recall = float(res[6][2])
    f1 = float(res[6][3])
    return  [acc, precision, recall, f1]

svm_df = get_df(report_svm)
lr_df = get_df(report_lr)
nb_df = get_df(report_nb)
rf_df = get_df(report_rf)
dt_df = get_df(report_dt)
ada_df = get_df(report_ada)
gb_df = get_df(report_gb)
sgd_df = get_df(report_sgd)
bag_df = get_df(report_bag)
df = pd.DataFrame([svm_df, lr_df, nb_df, rf_df,  dt_df, ada_df, gb_df, sgd_df, bag_df],
                    columns=['accuracy', 'precision', 'recall', 'f1-score'],
                    index=['svm', 'lr', 'nb', 'rf', 'dt', 'ada', 'gb',  'sgd', 'bag'])

In [ ]:
df

#### 0-> neg   |   1-> pos

In [ ]:
# use rf model to predict the test data

sen = ['???????', '?????']

def predict(sent,clf):
    X = [cut_words(sent)]
    X = vectorizer.transform(X)
    if type(sent) == str:
        return clf.predict(X)[0]
    else:
        result = []
        for i in range(len(sent)):
            result.append(clf.predict(X[i])[0])
        return result

def predict_muti(sen,clf_list):
    X = [cut_words(text) for text in sen]
    X = vectorizer.transform(X)
    result = []
    for clf in clf_list:
        res = []
        for i in range(len(sen)):
            res.append(clf.predict(X[i])[0])
        result.append(res)
    return result

model_list = [clf_SVC, clf_NB, clf_LR, clf_RF, clf_DT, clf_AdB, clf_GB, clf_SGD,clf_BG]
print(predict_muti(sen,model_list))
print(predict(sen[0],clf_RF))
